# Ch.4 — CI/CD Pipelines (GitHub Actions)

This notebook walks through setting up a complete CI/CD pipeline using GitHub Actions.

**Running example:** Auto-deploy Flask app on every push to main

**Prerequisites:**
- GitHub account
- Docker Hub account (free tier)
- Docker installed locally
- Kind installed (from Ch.3)

**What you'll build:**
- GitHub Actions workflow that runs tests
- Docker image build and push to Docker Hub
- Deployment to local Kind cluster
- Matrix builds (multi-version testing)
- Dependency caching for faster builds

## Cell 1: Set Up GitHub Actions in a Repo

First, create a sample Flask app and GitHub repository.

**Steps:**
1. Create a new directory for your project
2. Initialize a Flask app
3. Create a GitHub repository
4. Push code to GitHub

In [ ]:
import os
import subprocess

# Create project directory
project_dir = "flask-cicd-demo"
os.makedirs(project_dir, exist_ok=True)
os.chdir(project_dir)

# Create simple Flask app
app_code = '''
from flask import Flask, jsonify

app = Flask(__name__)

@app.route('/health')
def health():
    return jsonify({"status": "healthy"})

@app.route('/predict')
def predict():
    return jsonify({"prediction": "success"})

if __name__ == '__main__':
    app.run(host='0.0.0.0', port=5000)
'''

with open('app.py', 'w') as f:
    f.write(app_code)

# Create requirements.txt
with open('requirements.txt', 'w') as f:
    f.write('flask==3.0.0\n')

# Create Dockerfile
dockerfile = '''
FROM python:3.11-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
EXPOSE 5000
CMD ["python", "app.py"]
'''

with open('Dockerfile', 'w') as f:
    f.write(dockerfile)

# Initialize git repo
subprocess.run(['git', 'init'], check=True)
subprocess.run(['git', 'add', '.'], check=True)
subprocess.run(['git', 'commit', '-m', 'Initial commit'], check=True)

print("✅ Flask app created!")
print("\nNext steps:")
print("1. Create a new repo on GitHub")
print("2. git remote add origin <your-repo-url>")
print("3. git push -u origin main")

## Cell 2: Write a Simple Workflow (Run Tests)

Create your first GitHub Actions workflow that runs tests on every push.

In [ ]:
# Create tests directory
os.makedirs('tests', exist_ok=True)

# Create simple test file
test_code = '''
import pytest
from app import app

@pytest.fixture
def client():
    app.config['TESTING'] = True
    with app.test_client() as client:
        yield client

def test_health(client):
    response = client.get('/health')
    assert response.status_code == 200
    assert response.json['status'] == 'healthy'

def test_predict(client):
    response = client.get('/predict')
    assert response.status_code == 200
    assert 'prediction' in response.json
'''

with open('tests/test_app.py', 'w') as f:
    f.write(test_code)

# Add pytest to requirements
with open('requirements.txt', 'a') as f:
    f.write('pytest==7.4.3\n')

# Create GitHub Actions workflow directory
os.makedirs('.github/workflows', exist_ok=True)

# Create simple test workflow
workflow = '''
name: Run Tests

on:
  push:
    branches: [main]
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      
      - name: Install dependencies
        run: |
          pip install -r requirements.txt
      
      - name: Run tests
        run: pytest tests/ -v
'''

with open('.github/workflows/test.yml', 'w') as f:
    f.write(workflow)

print("✅ Test workflow created at .github/workflows/test.yml")
print("\nCommit and push to see it in action:")
print("git add .")
print("git commit -m 'Add test workflow'")
print("git push")

## Cell 3: Add Docker Build Step

Extend the workflow to build a Docker image after tests pass.

In [ ]:
# Updated workflow with Docker build
workflow_with_build = '''
name: CI Pipeline

on:
  push:
    branches: [main]
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      
      - name: Install dependencies
        run: pip install -r requirements.txt
      
      - name: Run tests
        run: pytest tests/ -v
  
  build:
    needs: test  # Only run if tests pass
    runs-on: ubuntu-latest
    if: github.ref == 'refs/heads/main'  # Only build on main branch
    steps:
      - uses: actions/checkout@v4
      
      - name: Set up Docker Buildx
        uses: docker/setup-buildx-action@v3
      
      - name: Build Docker image
        uses: docker/build-push-action@v5
        with:
          context: .
          push: false  # Don't push yet
          tags: flask-app:${{ github.sha }}
          load: true  # Load image for testing
      
      - name: Test Docker image
        run: |
          docker run -d -p 5000:5000 --name test-app flask-app:${{ github.sha }}
          sleep 5
          curl http://localhost:5000/health
          docker stop test-app
'''

with open('.github/workflows/ci.yml', 'w') as f:
    f.write(workflow_with_build)

print("✅ CI workflow with Docker build created!")
print("\nKey features:")
print("- Tests run first")
print("- Build job waits for tests to pass (needs: test)")
print("- Only builds on main branch")
print("- Tags image with commit SHA for traceability")

## Cell 4: Push Image to Docker Hub (Using Secrets)

Add Docker Hub authentication and push the built image.

**Before running:** Set up Docker Hub secrets in your GitHub repo:
1. Go to Settings → Secrets and variables → Actions
2. Add `DOCKER_USERNAME` (your Docker Hub username)
3. Add `DOCKER_TOKEN` (generate at hub.docker.com → Account Settings → Security → New Access Token)

In [ ]:
# Full CI/CD workflow with Docker Hub push
workflow_with_push = '''
name: CI/CD Pipeline

on:
  push:
    branches: [main]
  pull_request:

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
      - name: Install dependencies
        run: pip install -r requirements.txt
      - name: Run tests
        run: pytest tests/ -v
  
  build-and-push:
    needs: test
    runs-on: ubuntu-latest
    if: github.ref == 'refs/heads/main'
    steps:
      - uses: actions/checkout@v4
      
      - name: Set up Docker Buildx
        uses: docker/setup-buildx-action@v3
      
      - name: Log in to Docker Hub
        uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKER_USERNAME }}
          password: ${{ secrets.DOCKER_TOKEN }}
      
      - name: Build and push
        uses: docker/build-push-action@v5
        with:
          context: .
          push: true
          tags: |
            ${{ secrets.DOCKER_USERNAME }}/flask-app:latest
            ${{ secrets.DOCKER_USERNAME }}/flask-app:${{ github.sha }}
'''

with open('.github/workflows/ci-cd.yml', 'w') as f:
    f.write(workflow_with_push)

print("✅ Full CI/CD workflow created!")
print("\nSecrets required:")
print("- DOCKER_USERNAME: Your Docker Hub username")
print("- DOCKER_TOKEN: Docker Hub access token")
print("\nImage tagging strategy:")
print("- latest: Always points to most recent build")
print("- <commit-sha>: Permanent tag for rollback")

## Cell 5: Deploy to Local Kind Cluster from CI

Add a deployment step that updates a Kind cluster (useful for local testing).

In [ ]:
# Create Kubernetes deployment manifest
os.makedirs('k8s', exist_ok=True)

k8s_deployment = '''
apiVersion: apps/v1
kind: Deployment
metadata:
  name: flask-app
spec:
  replicas: 3
  selector:
    matchLabels:
      app: flask-app
  template:
    metadata:
      labels:
        app: flask-app
    spec:
      containers:
      - name: flask-app
        image: DOCKER_USERNAME/flask-app:latest  # Will be replaced by CI
        ports:
        - containerPort: 5000
---
apiVersion: v1
kind: Service
metadata:
  name: flask-app
spec:
  selector:
    app: flask-app
  ports:
  - port: 80
    targetPort: 5000
  type: LoadBalancer
'''

with open('k8s/deployment.yaml', 'w') as f:
    f.write(k8s_deployment)

# Add deploy job to workflow
deploy_job = '''
  deploy:
    needs: build-and-push
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      
      - name: Set up Kind cluster
        uses: helm/kind-action@v1.8.0
      
      - name: Update image tag in deployment
        run: |
          sed -i "s|DOCKER_USERNAME|${{ secrets.DOCKER_USERNAME }}|g" k8s/deployment.yaml
          sed -i "s|:latest|:${{ github.sha }}|g" k8s/deployment.yaml
      
      - name: Deploy to Kind
        run: |
          kubectl apply -f k8s/deployment.yaml
          kubectl rollout status deployment/flask-app --timeout=120s
      
      - name: Verify deployment
        run: |
          kubectl get pods
          kubectl get svc flask-app
'''

print("✅ Kubernetes manifests created in k8s/ directory")
print("\nDeploy job features:")
print("- Creates Kind cluster on CI runner")
print("- Updates deployment with new image tag")
print("- Waits for rollout to complete")
print("- Verifies pods are running")
print("\nNote: Add the deploy job from the output above to your workflow")

## Cell 6: Matrix Builds (Test on Multiple Python Versions)

Use matrix strategy to test your app on multiple Python versions in parallel.

In [ ]:
# Workflow with matrix strategy
matrix_workflow = '''
name: Matrix Build

on: [push, pull_request]

jobs:
  test:
    runs-on: ${{ matrix.os }}
    strategy:
      matrix:
        os: [ubuntu-latest, macos-latest, windows-latest]
        python-version: ['3.9', '3.10', '3.11', '3.12']
    steps:
      - uses: actions/checkout@v4
      
      - name: Set up Python ${{ matrix.python-version }}
        uses: actions/setup-python@v5
        with:
          python-version: ${{ matrix.python-version }}
      
      - name: Install dependencies
        run: pip install -r requirements.txt
      
      - name: Run tests
        run: pytest tests/ -v
'''

with open('.github/workflows/matrix.yml', 'w') as f:
    f.write(matrix_workflow)

print("✅ Matrix build workflow created!")
print("\nMatrix dimensions:")
print("- OS: ubuntu, macos, windows")
print("- Python: 3.9, 3.10, 3.11, 3.12")
print("- Total jobs: 3 × 4 = 12 (run in parallel)")
print("\n⚠️  This uses more CI minutes. For production:")
print("- Test only on ubuntu-latest")
print("- Test only Python versions you support")

## Cell 7: Caching Dependencies (Speed Up Builds)

Cache pip dependencies to avoid reinstalling them on every run.

In [ ]:
# Workflow with dependency caching
cached_workflow = '''
name: Cached Build

on: [push, pull_request]

jobs:
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      
      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: '3.11'
          cache: 'pip'  # Automatically cache pip dependencies
      
      - name: Install dependencies
        run: pip install -r requirements.txt
      
      - name: Run tests
        run: pytest tests/ -v
'''

with open('.github/workflows/cached.yml', 'w') as f:
    f.write(cached_workflow)

# Also show manual caching approach
print("✅ Cached workflow created!")
print("\nAutomatic caching (recommended):")
print("  cache: 'pip'  # Caches ~/.cache/pip")
print("\nManual caching (more control):")
print("""
- name: Cache pip packages
  uses: actions/cache@v3
  with:
    path: ~/.cache/pip
    key: ${{ runner.os }}-pip-${{ hashFiles('requirements.txt') }}
    restore-keys: |
      ${{ runner.os }}-pip-
""")
print("\n💡 Cache key includes requirements.txt hash")
print("   → Cache invalidates when dependencies change")

## Cell 8: Conditional Workflows (Deploy Only on Main Branch)

Use conditionals to control when jobs run.

In [ ]:
# Examples of conditional execution
conditionals = '''
# Example 1: Deploy only on main branch
deploy:
  runs-on: ubuntu-latest
  if: github.ref == 'refs/heads/main'
  steps: [...]

# Example 2: Skip job if commit message contains [skip ci]
test:
  runs-on: ubuntu-latest
  if: "!contains(github.event.head_commit.message, '[skip ci]')"
  steps: [...]

# Example 3: Run only on pull requests
lint:
  runs-on: ubuntu-latest
  if: github.event_name == 'pull_request'
  steps: [...]

# Example 4: Deploy to staging vs production
deploy-staging:
  if: github.ref == 'refs/heads/develop'
  steps:
    - name: Deploy to staging
      run: kubectl apply -f k8s/staging/

deploy-production:
  if: github.ref == 'refs/heads/main'
  steps:
    - name: Deploy to production
      run: kubectl apply -f k8s/production/

# Example 5: Manual approval for production
deploy-production:
  needs: test
  if: github.ref == 'refs/heads/main'
  environment:
    name: production
    url: https://myapp.com
  steps: [...]
'''

print("Common conditional patterns:")
print(conditionals)
print("\nUseful variables:")
print("- github.ref: Branch or tag ref (e.g., 'refs/heads/main')")
print("- github.event_name: 'push', 'pull_request', 'schedule'")
print("- github.actor: Username who triggered workflow")
print("- github.sha: Commit SHA")

## Cell 9: Manual Workflow Triggers (workflow_dispatch)

Allow manual workflow execution with custom inputs.

In [ ]:
# Workflow with manual trigger
manual_workflow = '''
name: Manual Deploy

on:
  workflow_dispatch:
    inputs:
      environment:
        description: 'Environment to deploy to'
        required: true
        type: choice
        options:
          - development
          - staging
          - production
      image_tag:
        description: 'Docker image tag to deploy'
        required: true
        default: 'latest'
      dry_run:
        description: 'Run in dry-run mode (no actual deployment)'
        type: boolean
        default: false

jobs:
  deploy:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      
      - name: Print inputs
        run: |
          echo "Environment: ${{ inputs.environment }}"
          echo "Image tag: ${{ inputs.image_tag }}"
          echo "Dry run: ${{ inputs.dry_run }}"
      
      - name: Deploy
        run: |
          if [ "${{ inputs.dry_run }}" == "true" ]; then
            echo "DRY RUN: Would deploy ${{ inputs.image_tag }} to ${{ inputs.environment }}"
          else
            echo "Deploying ${{ inputs.image_tag }} to ${{ inputs.environment }}..."
            # kubectl set image deployment/flask-app flask-app=${{ inputs.image_tag }}
          fi
'''

with open('.github/workflows/manual-deploy.yml', 'w') as f:
    f.write(manual_workflow)

print("✅ Manual workflow created!")
print("\nTo trigger manually:")
print("1. Go to GitHub repo → Actions tab")
print("2. Select 'Manual Deploy' workflow")
print("3. Click 'Run workflow' button")
print("4. Fill in inputs and run")
print("\nInput types supported:")
print("- string (default)")
print("- choice (dropdown)")
print("- boolean (checkbox)")
print("- environment (with approval gates)")

## Cell 10: View Logs and Artifacts

Access workflow logs and upload build artifacts.

In [ ]:
# Workflow with artifacts
artifacts_workflow = '''
name: Build with Artifacts

on: [push]

jobs:
  build:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      
      - name: Build application
        run: |
          mkdir -p build
          echo "Build completed at $(date)" > build/build-info.txt
          echo "Commit: ${{ github.sha }}" >> build/build-info.txt
      
      - name: Run tests and generate coverage
        run: |
          pip install -r requirements.txt pytest-cov
          pytest tests/ --cov=. --cov-report=html
      
      - name: Upload build artifacts
        uses: actions/upload-artifact@v4
        with:
          name: build-info
          path: build/
          retention-days: 30
      
      - name: Upload coverage report
        uses: actions/upload-artifact@v4
        with:
          name: coverage-report
          path: htmlcov/
          retention-days: 7
'''

with open('.github/workflows/artifacts.yml', 'w') as f:
    f.write(artifacts_workflow)

print("✅ Artifacts workflow created!")
print("\nArtifacts you can upload:")
print("- Build outputs (binaries, packages)")
print("- Test results (JUnit XML)")
print("- Coverage reports (HTML, Cobertura)")
print("- Logs (for debugging)")
print("\nTo download artifacts:")
print("1. Go to workflow run page")
print("2. Scroll to 'Artifacts' section")
print("3. Click artifact name to download")
print("\nViewing logs:")
print("- Click on any job in workflow run")
print("- Expand steps to see detailed output")
print("- Use search to filter log lines")
print("- Download raw logs (top right menu)")